# Content-based - похожие товары по характеристикам, лог. регрессия

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
import hdbscan
from sklearn.manifold import TSNE
import umap
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
import openpyxl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.compose import ColumnTransformer
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [2]:
#df - транзакционный очищенный датасет + кластеры каждого пользователя
#client_df - агрегированный датасет по пользователям
df = pd.read_parquet('df.parquet', engine='fastparquet')
client_df = pd.read_parquet('client_data.parquet', engine='fastparquet')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1393343 entries, 0 to 1393342
Data columns (total 40 columns):
 #   Column                    Non-Null Count    Dtype         
---  ------                    --------------    -----         
 0   Дата                      1393343 non-null  datetime64[ns]
 1   ДатаДоставки              1393343 non-null  datetime64[ns]
 2   НомерЗаказаНаСайте        1393343 non-null  object        
 3   НовыйСтатус               1393343 non-null  category      
 4   СуммаЗаказаНаСайте        1393343 non-null  float64       
 5   СуммаДокумента            1393343 non-null  float64       
 6   МетодДоставки             1393343 non-null  category      
 7   ФормаОплаты               1393343 non-null  category      
 8   Регион                    1379232 non-null  category      
 9   Группа2                   1393343 non-null  category      
 10  Группа3                   1393343 non-null  category      
 11  Группа4                   1335893 non-null  catego

In [4]:
client_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277805 entries, 0 to 277804
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Телефон_new        277805 non-null  object 
 1   orders_count       277805 non-null  float64
 2   items_total        277805 non-null  float64
 3   revenue_total      277805 non-null  float64
 4   avg_price          277805 non-null  float64
 5   margin_total       277805 non-null  float64
 6   unique_categories  277805 non-null  int64  
 7   recency_days       277805 non-null  int64  
 8   lifetime_days      277805 non-null  int64  
 9   avg_check          277805 non-null  float64
 10  items_per_order    277805 non-null  float64
 11  cluster_7          277805 non-null  int32  
dtypes: float64(7), int32(1), int64(3), object(1)
memory usage: 24.4+ MB


In [5]:
# Выбираем нужные колонки
df_cb = df[['Телефон_new', 'Номенклатура', 'ТипТовара', 'ДатаЗаказаНаСайте', 'cluster_7', 'Группа2', 'Группа3', 'Цена']].copy()

df_cb['cluster_7'] = df_cb['cluster_7'].astype('int32')
client_df['cluster_7'] = client_df['cluster_7'].astype('int32')

# Сортируем по времени
df_cb = df_cb.sort_values(['Телефон_new', 'ДатаЗаказаНаСайте'])

# Оставляем пользователей с >1 покупкой
user_counts = df_cb['Телефон_new'].value_counts()
valid_users = user_counts[user_counts > 1].index
df_cb = df_cb[df_cb['Телефон_new'].isin(valid_users)]

# Разделяем на train/test (последняя покупка в test)
test_df_cb = df_cb.groupby('Телефон_new').tail(1)
train_df_cb = df_cb.drop(test_df_cb.index)

print(f"Train size: {len(train_df_cb)}")
print(f"Test size: {len(test_df_cb)}")
print(f"Уникальных пользователей в test: {test_df_cb['Телефон_new'].nunique()}")

Train size: 1115608
Test size: 151475
Уникальных пользователей в test: 151475


In [6]:
#Создание негативных примеров# Получаем все уникальные товары

TOP_ITEMS = 5000
popular_items = train_df_cb['Номенклатура'].value_counts().head(TOP_ITEMS).index.tolist()

all_items = train_df_cb['Номенклатура'].unique()
print(f"Всего товаров: {len(all_items)}")

# Создаем позитивные примеры (последние покупки)
positive_df = train_df_cb[['Телефон_new', 'Номенклатура', 'cluster_7']].copy()
positive_df['label'] = 1
print(f"Позитивных примеров: {len(positive_df)}")

# Создаем негативные примеры (по 5 на каждого пользователя)
negative_samples = []

# Получаем уникальных пользователей
users = positive_df['Телефон_new'].unique()
print("Создание негативных примеров...")

for user in tqdm(users, desc="Обработка пользователей", unit="польз."):
    # Товары, которые пользователь уже покупал
    bought_items = set(train_df_cb[train_df_cb['Телефон_new'] == user]['Номенклатура'].unique())
    
    # Доступные для негатива товары (популярные, но не купленные)
    available_items = [item for item in all_items if item not in bought_items]
    
    # Берем 5 случайных
    import random
    if len(available_items) >= 5:
        neg_items = random.sample(available_items, 5)
    else:
        neg_items = available_items  # если меньше 5, берем все доступные
    
    cluster = positive_df[positive_df['Телефон_new'] == user]['cluster_7'].iloc[0]
    for item in neg_items:
        negative_samples.append([user, item, cluster, 0])

negative_df = pd.DataFrame(negative_samples, columns=['Телефон_new', 'Номенклатура', 'cluster_7', 'label'])
print(f"Негативных примеров: {len(negative_df)}")

# Объединяем
train_ml_df = pd.concat([positive_df, negative_df], ignore_index=True)
print(f"Итоговый размер обучающей выборки: {len(train_ml_df)}")

# Анализ покрытия тестовых товаров
print("\n" + "="*60)
print("АНАЛИЗ ПОКРЫТИЯ ТЕСТОВОЙ ВЫБОРКИ")
print("="*60)

test_items = test_df_cb['Номенклатура'].unique()
train_items = train_df_cb['Номенклатура'].unique()
new_items = set(test_items) - set(train_items)
items_in_popular = [item for item in test_items if item in popular_items]  # Изменено: теперь используем popular_items, а не жестко 1000

print(f"Всего товаров в тесте: {len(test_items)}")
print(f"Новых товаров (нет в train): {len(new_items)} ({len(new_items)/len(test_items)*100:.1f}%)")
print(f"Тестовых товаров в топ-{TOP_ITEMS}: {len(items_in_popular)} из {len(test_items)} ({len(items_in_popular)/len(test_items)*100:.1f}%)")
print("="*60)

Всего товаров: 120945
Позитивных примеров: 1115608
Создание негативных примеров...


Обработка пользователей: 100%|███| 151475/151475 [10:56:19<00:00,  3.85польз./s]


Негативных примеров: 757375
Итоговый размер обучающей выборки: 1872983

АНАЛИЗ ПОКРЫТИЯ ТЕСТОВОЙ ВЫБОРКИ
Всего товаров в тесте: 50281
Новых товаров (нет в train): 9882 (19.7%)
Тестовых товаров в топ-5000: 4764 из 50281 (9.5%)


In [7]:
# Добавляем характеристики товаров
item_features = df_cb[['Номенклатура', 'Группа2', 'Группа3', 'ТипТовара', 'Цена']].drop_duplicates('Номенклатура')

# Преобразуем категориальные колонки в строки перед заполнением
for col in ['Группа2', 'Группа3', 'ТипТовара']:
    item_features[col] = item_features[col].astype('object').fillna('unknown')

# Объединяем с train_ml_df
train_ml_df = train_ml_df.merge(item_features, on='Номенклатура', how='left')

# Заполняем пропуски
for col in ['Группа2', 'Группа3', 'ТипТовара']:
    train_ml_df[col] = train_ml_df[col].fillna('unknown')

train_ml_df['Цена'] = train_ml_df['Цена'].fillna(train_ml_df['Цена'].median())


In [8]:
# Берем фичи пользователей из client_df
user_features = client_df[['Телефон_new', 'orders_count', 'avg_check', 'recency_days', 'lifetime_days', 'unique_categories']].copy()

# Объединяем
train_ml_df = train_ml_df.merge(user_features, on='Телефон_new', how='left')

# Заполняем пропуски
train_ml_df['orders_count'] = train_ml_df['orders_count'].fillna(0)
train_ml_df['avg_check'] = train_ml_df['avg_check'].fillna(0)
train_ml_df['recency_days'] = train_ml_df['recency_days'].fillna(365)
train_ml_df['lifetime_days'] = train_ml_df['lifetime_days'].fillna(0)
train_ml_df['unique_categories'] = train_ml_df['unique_categories'].fillna(0)

In [9]:
#Добавляем исторические признаки
# Считаем, сколько раз пользователь покупал этот товар в прошлом
user_item_history = train_df_cb.groupby(['Телефон_new', 'Номенклатура']).size().reset_index(name='user_item_count')
train_ml_df = train_ml_df.merge(user_item_history, on=['Телефон_new', 'Номенклатура'], how='left')
train_ml_df['user_item_count'] = train_ml_df['user_item_count'].fillna(0)

# Считаем среднюю цену товаров, которые покупал пользователь
user_avg_price = train_df_cb.groupby('Телефон_new')['Цена'].mean().reset_index(name='user_avg_price')
train_ml_df = train_ml_df.merge(user_avg_price, on='Телефон_new', how='left')
train_ml_df['price_diff'] = train_ml_df['Цена'] - train_ml_df['user_avg_price']

In [10]:
# Кодируем категориальные признаки
# Сбрасываем индекс
train_ml_df = train_ml_df.reset_index(drop=True)

# Кодируем категориальные признаки
cat_cols = ['cluster_7', 'Группа2', 'Группа3', 'ТипТовара']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train_ml_df[col] = le.fit_transform(train_ml_df[col].values.astype(str))
    encoders[col] = le
    
# Выбираем фичи
feature_cols = cat_cols + ['Цена', 'orders_count', 'avg_check', 'recency_days', 'lifetime_days', 
                           'unique_categories', 'user_item_count', 'price_diff']

# Масштабируем числовые
scaler = StandardScaler()
numeric_cols = ['Цена', 'orders_count', 'avg_check', 'recency_days', 'lifetime_days', 
                'unique_categories', 'user_item_count', 'price_diff']
train_ml_df[numeric_cols] = scaler.fit_transform(train_ml_df[numeric_cols])

X = train_ml_df[feature_cols]
y = train_ml_df['label']

print(f"Признаки готовы. X shape: {X.shape}")
print(f"Баланс классов: {y.value_counts().to_dict()}")

Признаки готовы. X shape: (1872983, 12)
Баланс классов: {1: 1115608, 0: 757375}


In [11]:
# Обучаем логистическую регрессию с балансировкой классов
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X, y)

print(f"ROC-AUC на обучении: {roc_auc_score(y, model.predict_proba(X)[:, 1]):.4f}")

# Важность признаков
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)
print("\nТоп-10 важных признаков:")
print(feature_importance.head(10))

ROC-AUC на обучении: 1.0000

Топ-10 важных признаков:
              feature  coefficient
10    user_item_count    23.529669
0           cluster_7     0.650164
11         price_diff     0.612313
3           ТипТовара     0.271620
7        recency_days     0.262558
5        orders_count     0.258911
1             Группа2     0.147057
9   unique_categories     0.120007
6           avg_check     0.099554
2             Группа3     0.031948


In [12]:
def recommend_content_based(client_id, n=10):
    # --- 1. Проверка пользователя ---
    if client_id not in client_df['Телефон_new'].values:
        return popular_items[:n]

    user_row = client_df[client_df['Телефон_new'] == client_id]
    if user_row.empty:
        return popular_items[:n]
    user_row = user_row.iloc[0]

    # --- 2. Пользовательские признаки ---
    cluster = user_row['cluster_7']  # оставляем тип как есть, приведём к str при кодировании

    user_data = {
        'orders_count': user_row.get('orders_count', 0),
        'avg_check': user_row.get('avg_check', 0),
        'recency_days': user_row.get('recency_days', 365),
        'lifetime_days': user_row.get('lifetime_days', 0),
        'unique_categories': user_row.get('unique_categories', 0)
    }

    # --- 3. Средняя цена пользователя (как в обучении) ---
    # user_avg_price — это датафрейм из In[70] с колонками ['Телефон_new','user_avg_price']
    user_avg_price_row = user_avg_price[user_avg_price['Телефон_new'] == client_id]
    if not user_avg_price_row.empty:
        user_avg_price_val = user_avg_price_row['user_avg_price'].iloc[0]
    else:
        user_avg_price_val = user_data['avg_check']  # fallback

    # --- 4. История покупок ---
    bought_items = set(
        train_df_cb[train_df_cb['Телефон_new'] == client_id]['Номенклатура'].unique()
    )

    # --- 5. Кандидаты ---
    # товары из того же кластера
    items_same_cluster = train_df_cb[train_df_cb['cluster_7'] == cluster]['Номенклатура'].unique().tolist()

    # товары из тех же категорий (Группа2/Группа3/ТипТовара), что у пользователя в истории
    user_hist = train_df_cb[train_df_cb['Телефон_new'] == client_id]
    user_groups2 = set(user_hist['Группа2'].dropna().astype(str).unique())
    user_groups3 = set(user_hist['Группа3'].dropna().astype(str).unique())
    user_types = set(user_hist['ТипТовара'].dropna().astype(str).unique())

    items_same_cats = item_features[
        (item_features['Группа2'].astype(str).isin(user_groups2)) |
        (item_features['Группа3'].astype(str).isin(user_groups3)) |
        (item_features['ТипТовара'].astype(str).isin(user_types))
    ]['Номенклатура'].unique().tolist()

    # 3) объединяем кандидатов и убираем купленное
    candidate_items = list(dict.fromkeys(items_same_cluster + items_same_cats))
    candidate_items = [it for it in candidate_items if it not in bought_items]

    # 4) fallback на популярные
    if len(candidate_items) < 500:
        candidate_items += [it for it in popular_items if it not in bought_items]

    candidate_items = candidate_items[:500]


    # --- 6. Собираем признаки ---
    features_list = []
    for item in candidate_items:
        item_row = item_features[item_features['Номенклатура'] == item]
        if item_row.empty:
            continue

        item_row = item_row.iloc[0]
        price = item_row['Цена']

        row = {
            'Номенклатура': item,
            'cluster_7': str(cluster),
            'Группа2': str(item_row['Группа2']),
            'Группа3': str(item_row['Группа3']),
            'ТипТовара': str(item_row['ТипТовара']),
            'Цена': price,
            'orders_count': user_data['orders_count'],
            'avg_check': user_data['avg_check'],
            'recency_days': user_data['recency_days'],
            'lifetime_days': user_data['lifetime_days'],
            'unique_categories': user_data['unique_categories'],
            'user_item_count': 0,
            'price_diff': price - user_avg_price_val
        }
        features_list.append(row)

    if len(features_list) == 0:
        return popular_items[:n]

    candidates_df = pd.DataFrame(features_list)

    # --- 7. Кодирование категорий ---
    for col in cat_cols:
        le = encoders[col]
        candidates_df[col] = candidates_df[col].map(
            lambda x: le.transform([x])[0] if x in le.classes_ else -1
        )

    # --- 8. Масштабирование ---
    candidates_df[numeric_cols] = scaler.transform(candidates_df[numeric_cols])

    # --- 9. Предсказание ---
    X_candidates = candidates_df[feature_cols]
    probs = model.predict_proba(X_candidates)[:, 1]
    candidates_df['score'] = probs

    # --- 10. Сортировка ---
    recommendations = (
        candidates_df.sort_values('score', ascending=False)['Номенклатура']
        .head(n)
        .tolist()
    )
    return recommendations


In [13]:
def evaluate_content_based(test_df, recommender_func, k=10):
    hits = []
    precisions = []
    recalls = []
    average_precisions = []
    mrr = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Оценка (k={k})", unit="запрос"):
        user = row['Телефон_new']
        true_item = row['Номенклатура']
        
        try:
            recs = recommender_func(user, k)
        except Exception as e:
            continue
        
        hit = int(true_item in recs)
        hits.append(hit)
        precisions.append(hit / k)
        recalls.append(hit)
        
        if hit:
            rank = recs.index(true_item) + 1
            average_precisions.append(1 / rank)
            mrr.append(1 / rank)
        else:
            average_precisions.append(0)
            mrr.append(0)
    
    return {
        "HitRate@K": np.mean(hits),
        "Precision@K": np.mean(precisions),
        "Recall@K": np.mean(recalls),
        "MAP@K": np.mean(average_precisions),
        "MRR@K": np.mean(mrr),
        "Test_size": len(test_df)
    }

# Оценка
print("Подготовка тестовой выборки...")
test_sample = test_df_cb.sample(min(5000, len(test_df_cb)), random_state=42)
print(f"Размер тестовой выборки: {len(test_sample)}")

print("\nОценка моделей:")
metrics_cb_5 = evaluate_content_based(test_sample, recommend_content_based, k=5)
metrics_cb_10 = evaluate_content_based(test_sample, recommend_content_based, k=10)
metrics_cb_20 = evaluate_content_based(test_sample, recommend_content_based, k=20)

print("\n" + "="*50)
print("Content-based модель:")
print("="*50)
print(f"K=5: {metrics_cb_5}")
print(f"K=10: {metrics_cb_10}")
print(f"K=20: {metrics_cb_20}")

print("\n" + "="*50)
print("Сравнение с популярностной моделью:")
print("="*50)
print(f"Popularity HitRate@10: 0.0197")
print(f"Content-based HitRate@10: {metrics_cb_10['HitRate@K']:.4f}")
print(f"Улучшение: {((metrics_cb_10['HitRate@K'] - 0.0197) / 0.0197 * 100):.1f}%")

Подготовка тестовой выборки...
Размер тестовой выборки: 5000

Оценка моделей:


Оценка (k=20): 100%|██████████████████| 5000/5000 [2:52:50<00:00,  2.07s/запрос]


Content-based модель:
K=5: {'HitRate@K': np.float64(0.0008), 'Precision@K': np.float64(0.00016), 'Recall@K': np.float64(0.0008), 'MAP@K': np.float64(0.00055), 'MRR@K': np.float64(0.00055), 'Test_size': 5000}
K=10: {'HitRate@K': np.float64(0.0022), 'Precision@K': np.float64(0.00022), 'Recall@K': np.float64(0.0022), 'MAP@K': np.float64(0.0007105555555555556), 'MRR@K': np.float64(0.0007105555555555556), 'Test_size': 5000}
K=20: {'HitRate@K': np.float64(0.0032), 'Precision@K': np.float64(0.00016), 'Recall@K': np.float64(0.0032), 'MAP@K': np.float64(0.0007671929824561404), 'MRR@K': np.float64(0.0007671929824561404), 'Test_size': 5000}

Сравнение с популярностной моделью:
Popularity HitRate@10: 0.0197
Content-based HitRate@10: 0.0022
Улучшение: -88.8%


In [14]:
# Проверка покрытия тестовых товаров
test_items = test_df_cb['Номенклатура'].unique()
train_items = train_df_cb['Номенклатура'].unique()

new_items = set(test_items) - set(train_items)
print(f"Новых товаров в тесте: {len(new_items)} из {len(test_items)} ({len(new_items)/len(test_items)*100:.1f}%)")

# Проверка попадания в топ-1000
items_in_top1000 = [item for item in test_items if item in popular_items]
print(f"Тестовых товаров в топ-1000: {len(items_in_top1000)} из {len(test_items)} ({len(items_in_top1000)/len(test_items)*100:.1f}%)")

Новых товаров в тесте: 9882 из 50281 (19.7%)
Тестовых товаров в топ-1000: 4764 из 50281 (9.5%)


In [15]:
!jupyter nbconvert --to script ВКР_Zinoveva_3_content_based_1.ipynb

[NbConvertApp] Converting notebook ВКР_Zinoveva_3_content_based_1.ipynb to script
[NbConvertApp] Writing 15427 bytes to ВКР_Zinoveva_3_content_based_1.py


In [16]:
train_ml_df = train_ml_df.loc[:, ~train_ml_df.columns.duplicated()]